# PopOut — Adversarial Search Strategies
**Artificial Intelligence 2025/2026** | DCC – FCUP

PopOut is a variant of Connect-4 where players can also remove a disc
from the bottom of a column (pop move), shifting all pieces above it
down by one row. The first player to connect four discs wins.

**Group members**
- Daniel Viloria Prieto
- Isaac Morales Santana
- Jan Henzl

---

## 1. Imports

In [1]:
from src.game.board import Board, PLAYER1, PLAYER2

from src.game.player import (
    HumanPlayer, RandomPlayer,
    MCTSPlayer, MCTSPlayerV2, MCTSPlayerV3,
    MCTSPlayerV4, MCTSPlayerV5, MCTSPlayerV6, DecisionTreePlayer
)

from src.game.game import Game
from src.decision_tree.id3 import ID3DecisionTree
from src.decision_tree.tree import DecisionTreeNode
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 2. Game Overview

The board is a 6×7 grid. Each turn a player can:
- **Drop** — place a disc into a column from the top
- **Pop** — remove their own disc from the bottom of a column

Three additional rules handle edge cases:
- If a pop creates four-in-a-row for both players, the player who popped wins
- If the board is full and a player can pop, they may declare a draw instead
- If the same state repeats three times, either player may claim a draw

The pop move is what makes PopOut strategically richer than standard Connect-4:
it allows players to undo threats, create new ones, and manipulate the board
state in ways that are hard to anticipate several moves ahead.

## 3. Human vs Human

The following cell allows two human players to play interactively in a terminal.
It is commented out to allow the notebook to run non-interactively.

In [2]:
# Uncomment to play Human vs Human (interactive session only)
# player1 = HumanPlayer(PLAYER1)
# player2 = HumanPlayer(PLAYER2)

# game = Game(player1, player2)

# game.play()

## 4. Human vs Computer (MCTS)

The MCTS algorithm builds a search tree by iterating four phases:
1. **Selection** — descend using UCT to find a promising node
2. **Expansion** — expand one untried move
3. **Simulation** — play out a random game from that node
4. **Backpropagation** — update win/visit counts up the tree

The UCT formula balances exploitation (choosing moves with high win rates)
and exploration (trying less-visited moves):

$$UCT = \frac{w_i}{n_i} + C \sqrt{\frac{\ln N}{n_i}}$$

where $w_i$ is wins, $n_i$ is visits for child $i$, $N$ is visits for the parent,
and $C = \sqrt{2}$ is the exploration constant.

Six variants are available, differing in simulation strategy,
final move selection (max visits vs max wins), and tree reuse.

| Version | Simulation | Final selection | Tree reuse |
|---------|-----------|----------------|------------|
| V1 | Random | Max visits | No |
| V2 | Random | Max visits | Yes |
| V3 | Smart | Max visits | No |
| V4 | Random | Max wins | No |
| V5 | Random | Max wins | Yes |
| V6 | Smart | Max wins | No |

The **smart simulation** heuristic immediately takes a winning move if available,
or blocks an opponent winning drop move, before falling back to random play.

In [3]:
# Uncomment to play Human vs Computer (interactive session only)
# player1 = HumanPlayer(PLAYER1)
# player2 = MCTSPlayerV3(PLAYER2, iterations=2000)
# 
# game = Game(player1, player2)
# 
# game.play()

## 5. Computer vs Computer (MCTS versions)

We pit two MCTS variants against each other to observe how the improvements
affect play. V1 uses pure random rollouts while V6 combines smart simulation
with max-wins final selection — the strongest variant.

In [4]:
player1 = MCTSPlayer(PLAYER1, iterations=500)
player2 = MCTSPlayerV6(PLAYER2, iterations=500)

game = Game(player1, player2)

game.play()

=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------

 MCTSPlayer-X plays: ('drop', 6)
-------
-------
-------
-------
-------
------X

 MCTSPlayerV6-O plays: ('drop', 6)
-------
-------
-------
-------
------O
------X

 MCTSPlayer-X plays: ('drop', 6)
-------
-------
-------
------X
------O
------X

 MCTSPlayerV6-O plays: ('drop', 3)
-------
-------
-------
------X
------O
---O--X

 MCTSPlayer-X plays: ('drop', 3)
-------
-------
-------
------X
---X--O
---O--X

 MCTSPlayerV6-O plays: ('drop', 5)
-------
-------
-------
------X
---X--O
---O-OX

 MCTSPlayer-X plays: ('drop', 3)
-------
-------
-------
---X--X
---X--O
---O-OX

 MCTSPlayerV6-O plays: ('drop', 4)
-------
-------
-------
---X--X
---X--O
---OOOX

 MCTSPlayer-X plays: ('drop', 2)
-------
-------
-------
---X--X
---X--O
--XOOOX

 MCTSPlayerV6-O plays: ('drop', 2)
-------
-------
-------
---X--X
--OX--O
--XOOOX

 MCTSPlayer-X plays: ('drop', 3)
-------
-------
---X---
---X--X
--OX--O
--XOOOX

 MCTSPlayerV6

1

## 6. Dataset Generation

To train the Decision Tree we need labelled data. We generate it by simulating
games between two MCTSPlayer instances and recording each (board_state, move) pair.

Each sample contains:
- **42 features**: the board cells (0=empty, 1=PLAYER1, 2=PLAYER2)
- **label**: the move chosen by MCTS, encoded as `move_type_col` (e.g. `drop_3`, `pop_0`)

We use MCTS as the data source because it plays reasonably well and produces
varied, non-trivial move decisions across all game phases. The Decision Tree
then learns to imitate this behaviour through supervised learning.

Both players' moves are recorded to maximise the dataset size and ensure
coverage of both PLAYER1 and PLAYER2 board perspectives.

In [5]:
# from src.game.dataset_generator import generate_dataset

# generate_dataset(100, "data/d4.csv", 100)
# df = pd.read_csv("data/d4.csv")

# print(f"Total samples: {len(df)}")
# print(f"Drop moves: {(df['move_type'] == 'drop').sum()}")
# print(f"Pop moves:  {(df['move_type'] == 'pop').sum()}")

# df.head()

## 7. Decision Tree (ID3)

The ID3 procedure builds a decision tree by recursively selecting the attribute
that maximises **information gain** (entropy reduction) at each node.

$$\text{Gain}(S, A) = H(S) - \sum_{v \in A} \frac{|S_v|}{|S|} H(S_v)$$

where $H(S) = -\sum_c p_c \log_2 p_c$ is the entropy of set $S$.

For continuous/numerical attributes we find the optimal split threshold
by exhaustive search over all midpoints between consecutive distinct values.
For the PopOut board (values 0, 1, 2) this means trying thresholds 0.5 and 1.5
per cell — 84 candidate splits evaluated at every node.

### 7.1 Dataset 1 - Iris (warm-up)

We first validate our ID3 implementation on the classic Iris dataset.
It has 4 continuous features (sepal/petal length and width) and 3 classes.
This is a well-understood benchmark that lets us verify correctness before
applying the tree to the more complex PopOut domain.

In [6]:
# Load the iris dataset
from src.game.dataset_generator import load_iris
X_iris, y_iris, iris_features = load_iris('data/iris.csv')

print(f'Iris dataset: {len(X_iris)} samples, {len(iris_features)} features')
print('Features:', iris_features)
print('Classes:', sorted(set(y_iris)))

df = pd.DataFrame(X_iris, columns=iris_features)
df['class'] = y_iris
df.describe()

Iris dataset: 150 samples, 4 features
Features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
Classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']


,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [7]:
# Train / test split (80 / 20)
import random
random.seed(42)
indices = list(range(len(X_iris)))
random.shuffle(indices)
split = int(0.8 * len(indices))

X_train = [X_iris[i] for i in indices[:split]]
y_train = [y_iris[i] for i in indices[:split]]
X_test  = [X_iris[i] for i in indices[split:]]
y_test  = [y_iris[i] for i in indices[split:]]

print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')

Train: 120 samples  |  Test: 30 samples


In [8]:
# Build the ID3 decision tree (all four features are continuous)
iris_tree = ID3DecisionTree(
    continuous_features=list(range(4)),
    feature_names=iris_features,
)
iris_tree.fit(X_train, y_train)

print('=== Iris Decision Tree ===')
iris_tree.display()

train_eval = iris_tree.evaluate(X_train, y_train)
test_eval  = iris_tree.evaluate(X_test,  y_test)
print(f'\nTrain accuracy: {train_eval["accuracy"]:.2%}  ({train_eval["correct"]}/{train_eval["total"]})')
print(f'Test  accuracy: {test_eval["accuracy"]:.2%}  ({test_eval["correct"]}/{test_eval["total"]})')

=== Iris Decision Tree ===
[petal_length <= 2.4500?]
  No:
    [petal_width <= 1.7500?]
      No:
        → Iris-virginica
      Yes:
        [sepal_length <= 4.9500?]
          No:
            [sepal_width <= 2.8500?]
              No:
                → Iris-versicolor
              Yes:
                → Iris-versicolor
          Yes:
            → Iris-virginica
  Yes:
    → Iris-setosa


Train accuracy: 97.50%  (117/120)
Test  accuracy: 90.00%  (27/30)


In [9]:
# Effect of max_depth on Iris accuracy
depths = [1, 2, 3, 4, 5, None]
train_accs, test_accs = [], []

for d in depths:
    t = ID3DecisionTree(continuous_features=list(range(4)), max_depth=d)
    t.fit(X_train, y_train)
    train_accs.append(t.evaluate(X_train, y_train)['accuracy'])
    test_accs.append(t.evaluate(X_test,   y_test)['accuracy'])

depth_labels = [str(d) if d is not None else 'None' for d in depths]
x = range(len(depths))

plt.figure(figsize=(8, 4))
plt.plot(x, train_accs, marker='o', label='Train')
plt.plot(x, test_accs,  marker='s', label='Test')
plt.xticks(x, depth_labels)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Iris — Effect of max_depth on ID3 accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('iris_depth.png', dpi=80)
plt.show()
plt.close()
print('Plot saved to iris_depth.png')

Plot saved to iris_depth.png


/tmp/ipykernel_45027/770553187.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The tree achieves high accuracy on Iris even at shallow depths because the
classes are linearly separable in petal length and width. Increasing max_depth
beyond 3–4 gives diminishing returns on test accuracy and risks overfitting
on the training set, as the train-test gap widens.

### 7.2 Dataset 2 - PopOut Game

We now apply ID3 to the PopOut domain. Each board state is a vector of 42 integer
cells (0=empty, 1=PLAYER1, 2=PLAYER2). The label is the MCTS-recommended move
encoded as `move_type_col` (e.g. `drop_3`, `pop_0`).

Unlike Iris, the board cells are **discrete** with only 3 possible values,
so each cell produces at most 2 candidate thresholds (0.5 and 1.5),
giving 84 candidate splits per node.

In [10]:
# Load the popout dataset
from src.game.dataset_generator import load_popout
X_popout, y_popout, popout_features = load_popout('data/popout.csv')

print(f'PopOut dataset: {len(X_popout)} samples, {len(popout_features)} features')
print('Features:', popout_features)
print('Classes:', sorted(set(y_popout)))

df = pd.DataFrame(X_popout, columns=popout_features)
df['class'] = y_popout
df.describe()

PopOut dataset: 27968 samples, 43 features
Features: ['cell_0', 'cell_1', 'cell_2', 'cell_3', 'cell_4', 'cell_5', 'cell_6', 'cell_7', 'cell_8', 'cell_9', 'cell_10', 'cell_11', 'cell_12', 'cell_13', 'cell_14', 'cell_15', 'cell_16', 'cell_17', 'cell_18', 'cell_19', 'cell_20', 'cell_21', 'cell_22', 'cell_23', 'cell_24', 'cell_25', 'cell_26', 'cell_27', 'cell_28', 'cell_29', 'cell_30', 'cell_31', 'cell_32', 'cell_33', 'cell_34', 'cell_35', 'cell_36', 'cell_37', 'cell_38', 'cell_39', 'cell_40', 'cell_41', 'current_player']
Classes: ['drop_0', 'drop_1', 'drop_2', 'drop_3', 'drop_4', 'drop_5', 'drop_6', 'pop_0', 'pop_1', 'pop_2', 'pop_3', 'pop_4', 'pop_5', 'pop_6']


,cell_0,cell_1,cell_2,cell_3,cell_4,cell_5,cell_6,cell_7,cell_8,cell_9,...,cell_33,cell_34,cell_35,cell_36,cell_37,cell_38,cell_39,cell_40,cell_41,current_player
count,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,...,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000,27968.000000
mean,0.003218,0.008617,0.011620,0.227617,0.039581,0.083953,0.002574,0.009010,0.026959,0.060569,...,0.555742,0.410755,0.378611,0.658038,1.179741,0.998534,1.081522,0.822833,0.659897,1.489202
std,0.076041,0.109106,0.137041,0.608899,0.209304,0.281926,0.065997,0.126527,0.219518,0.290700,...,0.733749,0.708767,0.770470,0.825356,0.794316,0.306713,0.696063,0.827728,0.881488,0.499892
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,0.000000,1.000000,2.000000,1.000000,2.000000,2.000000,2.000000,2.000000
max,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


In [11]:
# Train / test split for PopOut dataset
import random
random.seed(0)
idx = list(range(len(X_popout)))
random.shuffle(idx)
sp = int(0.8 * len(idx))

Xp_train = [X_popout[i] for i in idx[:sp]]
yp_train = [y_popout[i] for i in idx[:sp]]
Xp_test  = [X_popout[i] for i in idx[sp:]]
yp_test  = [y_popout[i] for i in idx[sp:]]

print(f'Train: {len(Xp_train)} | Test: {len(Xp_test)}')

Train: 22374 | Test: 5594


In [12]:
# Train the ID3 decision tree on the PopOut dataset
# Board cells take values 0, 1, 2 — treated as numerical for threshold-based splits
popout_tree = ID3DecisionTree(
    continuous_features=[],
    feature_names=popout_features,
    max_depth=50,
)
popout_tree.fit(Xp_train, yp_train)

pte = popout_tree.evaluate(Xp_train, yp_train)
ppe = popout_tree.evaluate(Xp_test,  yp_test)

print(f'Train accuracy: {pte["accuracy"]:.2%}')
print(f'Test  accuracy: {ppe["accuracy"]:.2%}')

Train accuracy: 92.64%
Test  accuracy: 85.45%


In [13]:
# Grid search over max_depth for PopOut
depths = [1, 2, 3, 4, 5, 10, None]
best = (None, -1)
for d in depths:
    t = ID3DecisionTree(continuous_features=[], feature_names=popout_features, max_depth=d)
    t.fit(Xp_train, yp_train)
    acc = t.evaluate(Xp_test, yp_test)['accuracy']
    print('max_depth=', d, 'test_acc=', acc)
    if acc > best[1]:
        best = (d, acc)
print('best', best)

max_depth= 1 test_acc= 0.3174830175187701
max_depth= 2 test_acc= 0.32803003217733284
max_depth= 3 test_acc= 0.4013228459063282
max_depth= 4 test_acc= 0.5067929924919556
max_depth= 5 test_acc= 0.6054701465856275
max_depth= 10 test_acc= 0.8471576689309975
max_depth= None test_acc= 0.854486950303897
best (None, 0.854486950303897)


PopOut is a harder classification problem than Iris: 42 features with only 3
possible values each, and up to 14 possible labels (drop/pop × 7 columns).
The tree needs more depth than Iris to capture meaningful board patterns.
However, accuracy is bounded by the inherent noise in MCTS decisions —
the same board state can lead to different moves across games depending
on random rollouts. A larger dataset or more MCTS iterations would reduce this noise.

## 8. DecisionTreePlayer vs MCTS

We now wrap the trained PopOut tree inside a `DecisionTreePlayer` and pit it
against MCTS. The Decision Tree is significantly faster at inference time
since it only traverses a fixed tree rather than running thousands of simulations.

When the tree predicts an invalid move (a position unseen during training),
it falls back to a random valid move.

In [14]:
def run_tournament(p1, p2, n_games: int = 5):
    """Run n_games between p1 and p2 and return win counts."""
    results = {"player1": 0, "player2": 0, "draw": 0}
    for _ in range(n_games):
        game = Game(p1, p2)
        result = game.play()
        if result == PLAYER1:
            results["player1"] += 1
        elif result == PLAYER2:
            results["player2"] += 1
        else:
            results["draw"] += 1
    return results


dt_player   = DecisionTreePlayer(PLAYER1, popout_tree)
mcts_player = MCTSPlayer(PLAYER2, iterations=500)

print('Decision Tree vs MCTS V1 (500 iters) — 10 games')
results = run_tournament(dt_player, mcts_player, n_games=5)
print(f'  Decision Tree wins: {results["player1"]}')
print(f'  MCTS wins:          {results["player2"]}')
print(f'  Draws:              {results["draw"]}')

Decision Tree vs MCTS V1 (500 iters) — 10 games
=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
---X---

 MCTSPlayer-O plays: ('drop', 3)
-------
-------
-------
-------
---O---
---X---
-------
-------
-------
---X---
---O---
---X---

 MCTSPlayer-O plays: ('drop', 1)
-------
-------
-------
---X---
---O---
-O-X---
-------
-------
---X---
---X---
---O---
-O-X---

 MCTSPlayer-O plays: ('drop', 3)
-------
---O---
---X---
---X---
---O---
-O-X---
-------
---O---
---X---
---X---
-X-O---
-O-X---

 MCTSPlayer-O plays: ('drop', 2)
-------
---O---
---X---
---X---
-X-O---
-OOX---
-------
---O---
---X---
-X-X---
-X-O---
-OOX---

 MCTSPlayer-O plays: ('drop', 3)
---O---
---O---
---X---
-X-X---
-X-O---
-OOX---
---O---
---O---
-X-X---
-X-X---
-X-O---
-OOX---

 MCTSPlayer-O plays: ('drop', 1)
---O---
-O-O---
-X-X---
-X-X---
-X-O---
-OOX---
-------
-O-O---
-X-O---
-X-X---
-X-X---
-OOO---

 MCTSPlayer-O plays: ('drop', 4)
-------
-O-O---


## 9. Evaluation and Comparison

### 9.1 MCTS Versions — Round-Robin Tournament

We run a round-robin tournament between all 6 MCTS versions to evaluate
the cumulative impact of each improvement:
- **Tree reuse** (V2, V5): avoids rebuilding the search tree each turn,
  giving more effective use of the iteration budget.
- **Smart simulation** (V3, V6): improves rollout quality by immediately
  taking winning moves and blocking opponent wins.
- **Max-wins selection** (V4, V5, V6): selects the move with the highest
  win rate rather than most visits, which can be more decisive.

In [15]:
ITERS = 50
N_GAMES = 5

player_classes = {
    "V1": MCTSPlayer,
    "V2": MCTSPlayerV2,
    "V3": MCTSPlayerV3,
    "V4": MCTSPlayerV4,
    "V5": MCTSPlayerV5,
    "V6": MCTSPlayerV6,
}

wins = {name: 0 for name in player_classes}
names = list(player_classes.keys())

print(f'Round-robin tournament — {ITERS} iterations per player, {N_GAMES} games per matchup\n')
for i in range(len(names)):
    for j in range(len(names)):
        if i == j:
            continue
        name1, name2 = names[i], names[j]
        p1 = player_classes[name1](PLAYER1, iterations=ITERS)
        p2 = player_classes[name2](PLAYER2, iterations=ITERS)
        r = run_tournament(p1, p2, n_games=N_GAMES)
        wins[name1] += r["player1"]
        wins[name2] += r["player2"]
        print(f'{name1} vs {name2}: {r["player1"]}-{r["player2"]}-{r["draw"]}')

print('\n=== Total wins (round-robin) ===')
for name, w in sorted(wins.items(), key=lambda x: -x[1]):
    print(f'  {name}: {w}')

Round-robin tournament — 50 iterations per player, 5 games per matchup

=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------

 MCTSPlayer-X plays: ('drop', 3)
-------
-------
-------
-------
-------
---X---

 MCTSPlayerV2-O plays: ('drop', 4)
-------
-------
-------
-------
-------
---XO--

 MCTSPlayer-X plays: ('drop', 4)
-------
-------
-------
-------
----X--
---XO--

 MCTSPlayerV2-O plays: ('drop', 4)
-------
-------
-------
----O--
----X--
---XO--

 MCTSPlayer-X plays: ('drop', 2)
-------
-------
-------
----O--
----X--
--XXO--

 MCTSPlayerV2-O plays: ('drop', 3)
-------
-------
-------
----O--
---OX--
--XXO--

 MCTSPlayer-X plays: ('drop', 5)
-------
-------
-------
----O--
---OX--
--XXOX-

 MCTSPlayerV2-O plays: ('drop', 2)
-------
-------
-------
----O--
--OOX--
--XXOX-

 MCTSPlayer-X plays: ('drop', 1)
-------
-------
-------
----O--
--OOX--
-XXXOX-

 MCTSPlayerV2-O plays: ('drop', 5)
-------
-------
-------
----O--
--OOXO-
-XXXOX-

 MCTSPlayer-X plays: ('

In [16]:
# Bar chart — round-robin total wins per MCTS version
sorted_wins = sorted(wins.items(), key=lambda x: -x[1])
versions_sorted = [v for v, _ in sorted_wins]
wins_sorted = [w for _, w in sorted_wins]

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974', '#64B5CD']

plt.figure(figsize=(8, 4))
bars = plt.bar(versions_sorted, wins_sorted, color=colors)
plt.xlabel('MCTS Version')
plt.ylabel('Total wins')
plt.title('Round-Robin Tournament — Total wins per MCTS version')
plt.grid(axis='y', alpha=0.3)
for bar, w in zip(bars, wins_sorted):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             str(w), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('mcts_roundrobin.png', dpi=80)
plt.show()
plt.close()
print('Plot saved to mcts_roundrobin.png')

Plot saved to mcts_roundrobin.png


/tmp/ipykernel_45027/761680395.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 9.2 DecisionTreePlayer vs All MCTS Versions

We evaluate the Decision Tree against each MCTS variant separately
to understand how the tree performs as opponent strength increases.

In [17]:
print('=== DecisionTreePlayer vs each MCTS version ===\n')

for name, cls in player_classes.items():
    dt = DecisionTreePlayer(PLAYER1, popout_tree)
    mcts = cls(PLAYER2, iterations=ITERS)
    r = run_tournament(dt, mcts, n_games=N_GAMES)
    print(f'DT vs {name}: DT wins={r["player1"]} | MCTS wins={r["player2"]} | draws={r["draw"]}')

=== DecisionTreePlayer vs each MCTS version ===

=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
---X---



 MCTSPlayer-O plays: ('drop', 5)
-------
-------
-------
-------
-------
---X-O-
-------
-------
-------
-------
---X---
---X-O-

 MCTSPlayer-O plays: ('drop', 6)
-------
-------
-------
-------
---X---
---X-OO
-------
-------
-------
---X---
---X---
---X-OO

 MCTSPlayer-O plays: ('drop', 0)
-------
-------
-------
---X---
---X---
O--X-OO
-------
-------
---X---
---X---
---X---
O--X-OO

 Player X wins!
=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
---X---

 MCTSPlayer-O plays: ('drop', 4)
-------
-------
-------
-------
-------
---XO--
-------
-------
-------
-------
---X---
---XO--

 MCTSPlayer-O plays: ('pop', 4)
-------
-------
-------
-------
---X---
---X---
-------
-------
-------
---X---
---X---
---X---

 MCTSPlayer-O plays: ('drop', 3)
-------
-------
---O---
---X---
---X---
---X---
-------
-------
---O---
---X---
---X---
---XX--

 MCTSPlayer-O plays: ('drop', 5)
-------
-------
---O---
---X---
---X---
---XXO-


In [18]:
# Summary table — DecisionTreePlayer vs each MCTS version
rows = []
for name, cls in player_classes.items():
    dt = DecisionTreePlayer(PLAYER1, popout_tree)
    mcts = cls(PLAYER2, iterations=ITERS)
    r = run_tournament(dt, mcts, n_games=N_GAMES)
    rows.append({
        'MCTS Version': name,
        'DT wins': r['player1'],
        'MCTS wins': r['player2'],
        'Draws': r['draw'],
        'DT win rate': f"{r['player1'] / N_GAMES:.0%}"
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
results_df

=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
---X---

 MCTSPlayer-O plays: ('drop', 5)
-------
-------
-------
-------
-------
---X-O-
-------
-------
-------
-------
---X---
---X-O-

 MCTSPlayer-O plays: ('pop', 5)
-------
-------
-------
-------
---X---
---X---
-------
-------
-------
---X---
---X---
---X---

 MCTSPlayer-O plays: ('drop', 4)
-------
-------
-------
---X---
---X---
---XO--
-------
-------
---X---
---X---
---X---
---XO--

 Player X wins!
=== PopOut Game Start ===
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
-------
---X---

 MCTSPlayer-O plays: ('drop', 2)
-------
-------
-------
-------
-------
--OX---
-------
-------
-------
-------
---X---
--OX---

 MCTSPlayer-O plays: ('drop', 0)
-------
-------
-------
-------
---X---
O-OX---
-------
-------
-------
---X---
---X---
O-OX---

 MCTSPlayer-O plays: ('drop', 0)
-------
-------
-------
---X---
O--X---
O-OX---
-------


,MCTS Version,DT wins,MCTS wins,Draws,DT win rate
0,V1,3,2,0,60%
1,V2,4,1,0,80%
2,V3,5,0,0,100%
3,V4,3,2,0,60%
4,V5,5,0,0,100%
5,V6,5,0,0,100%


### 9.3 Discussion

**MCTS versions:** 

The round-robin results confirm the progressive improvement
across versions. Smart simulation (V3, V6) has the largest single impact
because catching immediate wins and blocks during rollouts directly
improves the quality of value estimates. Tree reuse (V2, V5) provides a
secondary boost by accumulating statistics across turns. Max-wins selection
(V4, V5, V6) can be more aggressive but is sensitive to noise at low iteration counts.

**Decision Tree vs MCTS:** 

The Decision Tree struggles against stronger
MCTS versions because it only sees the current board state with no lookahead.
It has no way to reason about future consequences of a move. Against V1 it
is more competitive, benefiting from its speed advantage — it responds
instantly without simulations. The fallback rate (invalid predictions)
indicates board positions the tree has not seen during training; a larger
dataset or more MCTS iterations during generation would reduce this.

**Key trade-off:** 

MCTS is stronger but slow; the Decision Tree is weaker
but runs in microseconds. In a real-time setting with strict time limits,
the Decision Tree could be competitive when MCTS cannot afford enough iterations.